In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

from ppghr.features import subject_features, FEATURE_NAMES
from ppghr.io import ACTIVITIES

t0 = time.time()
X1, y1, a1 = subject_features(1)
X2, y2, a2 = subject_features(2)
print(f"{time.time()-t0:.1f}s for 2 subjects")
print(f"X1 {X1.shape}  y1 {y1.shape}  features {len(FEATURE_NAMES)}")
print(f"NaNs: {np.isnan(X1).sum()}   infs: {np.isinf(X1).sum()}")
print(f"baseline MAE on S1 from features: {np.abs(X1[:, -2] - y1).mean():.2f}")

11.6s for 2 subjects
X1 (4603, 21)  y1 (4603,)  features 21
NaNs: 0   infs: 0
baseline MAE on S1 from features: 9.09


In [2]:
import pickle
from pathlib import Path
from ppghr.io import subject_ids, PROJECT_ROOT

cache = PROJECT_ROOT / "data" / "processed" / "features.pkl"

if cache.exists():
    with open(cache, "rb") as f:
        d = pickle.load(f)
    X, y, act, groups = d["X"], d["y"], d["act"], d["groups"]
else:
    ids = subject_ids()
    print(f"subjects: {ids}")
    Xs, ys, acts, gs = [], [], [], []
    t0 = time.time()
    for sid in ids:
        Xi, yi, ai = subject_features(sid)
        Xs.append(Xi); ys.append(yi); acts.append(ai)
        gs.append(np.full(len(yi), sid))
        print(f"  S{sid:<2} {len(yi):5d} windows   baseline MAE {np.abs(Xi[:,-2]-yi).mean():5.2f}")
    X = np.vstack(Xs); y = np.concatenate(ys)
    act = np.concatenate(acts); groups = np.concatenate(gs)
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump({"X": X, "y": y, "act": act, "groups": groups}, f)
    print(f"\n{time.time()-t0:.0f}s total")

print(f"\nX {X.shape}   y {y.shape}   subjects {len(np.unique(groups))}")
print(f"baseline MAE, all subjects: {np.abs(X[:, -2] - y).mean():.2f} bpm")

subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  S1   4603 windows   baseline MAE  9.09
  S2   4099 windows   baseline MAE 10.29
  S3   4367 windows   baseline MAE 11.34
  S4   4572 windows   baseline MAE 17.66
  S5   4649 windows   baseline MAE 31.96
  S6   2622 windows   baseline MAE 14.34
  S7   4668 windows   baseline MAE  5.45
  S8   4037 windows   baseline MAE 13.71
  S9   4277 windows   baseline MAE 20.89
  S10  5321 windows   baseline MAE 10.66
  S11  4521 windows   baseline MAE 23.46
  S12  3954 windows   baseline MAE 11.34
  S13  4565 windows   baseline MAE  8.57
  S14  4476 windows   baseline MAE  9.82
  S15  3966 windows   baseline MAE 11.58

82s total

X (64697, 21)   y (64697,)   subjects 15
baseline MAE, all subjects: 14.02 bpm


In [3]:
from sklearn.model_selection import LeaveOneGroupOut
from xgboost import XGBRegressor

def loso_eval(X, y, groups, feature_idx=None, **kw):
    """Leave-one-subject-out CV. Returns predictions aligned to y."""
    cols = slice(None) if feature_idx is None else feature_idx
    pred = np.zeros(len(y))
    logo = LeaveOneGroupOut()
    for tr, te in logo.split(X, y, groups):
        m = XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="reg:absoluteerror", n_jobs=-1, **kw
        )
        m.fit(X[tr][:, cols], y[tr])
        pred[te] = m.predict(X[te][:, cols])
        sid = groups[te][0]
        print(f"  held out S{sid:<2}  MAE {np.abs(pred[te]-y[te]).mean():5.2f}")
    return pred

t0 = time.time()
pred_full = loso_eval(X, y, groups)
print(f"\n{time.time()-t0:.0f}s")
print(f"XGBoost LOSO MAE: {np.abs(pred_full - y).mean():.2f} bpm")
print(f"baseline  LOSO MAE: {np.abs(X[:, -2] - y).mean():.2f} bpm")

  held out S1   MAE  8.20
  held out S2   MAE  5.90
  held out S3   MAE  4.91
  held out S4   MAE  7.29
  held out S5   MAE 21.51
  held out S6   MAE  8.34
  held out S7   MAE  3.87
  held out S8   MAE 11.75
  held out S9   MAE  8.88
  held out S10  MAE  5.52
  held out S11  MAE 10.78
  held out S12  MAE 14.04
  held out S13  MAE  4.87
  held out S14  MAE  4.40
  held out S15  MAE  5.48

55s
XGBoost LOSO MAE: 8.34 bpm
baseline  LOSO MAE: 14.02 bpm


In [4]:
names = list(FEATURE_NAMES)
acc_only = [i for i, n in enumerate(names) if n.startswith("acc")]
ppg_only = [i for i, n in enumerate(names) if n.startswith("ppg")]
no_temporal = [i for i, n in enumerate(names) if not n.endswith("_bpm") or n.startswith(("ppg_peak", "acc"))]

print("ACC-only features:", [names[i] for i in acc_only], "\n")
pred_acc = loso_eval(X, y, groups, feature_idx=acc_only)
print(f"\nACC-only LOSO MAE: {np.abs(pred_acc - y).mean():.2f} bpm")

ACC-only features: ['accx_peak_bpm', 'accy_peak_bpm', 'accz_peak_bpm', 'accx_peak_pow', 'accy_peak_pow', 'accz_peak_pow', 'acc_total_energy'] 

  held out S1   MAE 13.76
  held out S2   MAE  8.50
  held out S3   MAE  9.08
  held out S4   MAE  8.30
  held out S5   MAE 38.02
  held out S6   MAE 29.90
  held out S7   MAE  9.31
  held out S8   MAE 14.44
  held out S9   MAE  8.99
  held out S10  MAE 10.50
  held out S11  MAE 18.63
  held out S12  MAE 17.86
  held out S13  MAE 10.64
  held out S14  MAE  9.61
  held out S15  MAE 10.25

ACC-only LOSO MAE: 14.13 bpm


In [5]:
pred_ppg = loso_eval(X, y, groups, feature_idx=ppg_only)
print(f"\nPPG-only LOSO MAE: {np.abs(pred_ppg - y).mean():.2f} bpm")

  held out S1   MAE 11.25
  held out S2   MAE  6.89
  held out S3   MAE  6.92
  held out S4   MAE  7.99
  held out S5   MAE 27.23
  held out S6   MAE 15.19
  held out S7   MAE  4.61
  held out S8   MAE 12.30
  held out S9   MAE 12.53
  held out S10  MAE  7.31
  held out S11  MAE 13.72
  held out S12  MAE 15.27
  held out S13  MAE  6.53
  held out S14  MAE  5.62
  held out S15  MAE  7.55

PPG-only LOSO MAE: 10.57 bpm


In [6]:
import json

results = {
    "baseline_loso": float(np.abs(X[:, -2] - y).mean()),
    "xgb_full_loso": float(np.abs(pred_full - y).mean()),
    "xgb_acc_only_loso": float(np.abs(pred_acc - y).mean()),
    "xgb_ppg_only_loso": float(np.abs(pred_ppg - y).mean()),
    "constant_predictor": float(np.abs(y - y.mean()).mean()),
    "n_windows": int(len(y)),
    "n_subjects": int(len(np.unique(groups))),
    "published_classical": 11.06,
    "published_cnn": 7.65,
}
per_subject = {
    int(s): {
        "baseline": float(np.abs(X[groups == s, -2] - y[groups == s]).mean()),
        "xgb": float(np.abs(pred_full[groups == s] - y[groups == s]).mean()),
        "acc_only": float(np.abs(pred_acc[groups == s] - y[groups == s]).mean()),
        "ppg_only": float(np.abs(pred_ppg[groups == s] - y[groups == s]).mean()),
        "n": int((groups == s).sum()),
    }
    for s in np.unique(groups)
}

out = PROJECT_ROOT / "results" / "metrics"
out.mkdir(parents=True, exist_ok=True)
with open(out / "loso_results.json", "w") as f:
    json.dump({"overall": results, "per_subject": per_subject}, f, indent=2)

np.savez(PROJECT_ROOT / "results" / "metrics" / "predictions.npz",
         y=y, groups=groups, act=act,
         pred_full=pred_full, pred_acc=pred_acc, pred_ppg=pred_ppg,
         baseline=X[:, -2])

print(json.dumps(results, indent=2))

{
  "baseline_loso": 14.023264468426197,
  "xgb_full_loso": 8.336813995349466,
  "xgb_acc_only_loso": 14.129283828708502,
  "xgb_ppg_only_loso": 10.57469513480127,
  "constant_predictor": 17.672518335723556,
  "n_windows": 64697,
  "n_subjects": 15,
  "published_classical": 11.06,
  "published_cnn": 7.65
}
